# Ch12 Agent & RAG：构建自主智能体 教案

**课程名称：** Agent & RAG：构建自主智能体

**预计总时长：** 80-90 分钟

**源文件：** `Ch12_Agent_RAG/Ch12_Agent_RAG.ipynb`（共 27 个 Cell，Cell 0-26）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 环境准备 + Agent 架构全景图 | Cell 0-4 | 8 min |
| 8-22 min | ReAct 范式：理论 + 示例演示 | Cell 5-7 | 14 min |
| 22-35 min | Tool Use：工具定义与调用 | Cell 8-10 | 13 min |
| 35-40 min | **休息 + 回顾** | -- | 5 min |
| 40-55 min | Agent 实现：Function Calling + ProductionAgent | Cell 11-13 | 15 min |
| 55-70 min | RAG 理论 + 向量检索实现 + 可视化 | Cell 14-18 | 15 min |
| 70-75 min | **休息 + 回顾** | -- | 5 min |
| 75-82 min | Agent + RAG 结合 + 总结 | Cell 19-22 | 7 min |
| 82-90 min | 练习：余弦相似度 + 讨论 | Cell 23-26 | 8 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 已安装（`import json, re, numpy; print('OK')`）
- [ ] 确认 numpy 和 matplotlib 已安装
- [ ] 确认中文字体文件 `assets/fonts/NotoSansCJKsc-Regular.otf` 存在（可视化需要）
- [ ] 本章不需要 GPU，纯 CPU 即可
- [ ] 本章不需要真实 LLM API，所有 Agent 行为均为规则模拟
- [ ] 预跑一遍全部 Cell，确认所有输出正常（无网络依赖）
- [ ] 准备白板或画板，用于手绘 ReAct 循环和 RAG 流程图

---

## 第一段：开场 + 环境准备 + Agent 架构全景图（Cell 0-4）

📍 运行 Cell 0-1（Markdown 导读）、Cell 3（环境准备代码）、Cell 4（Agent 架构全景图）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：LLM 有什么局限？为什么需要 Agent 和 RAG？
- 确认环境就绪
- 通过全景图建立 Agent 系统的全局观

🗣 讲课话术

> 大家好！前面所有章节我们把 LLM 从零搭建、训练、对齐，模型已经很强了。但今天我们来聊一个关键问题：**LLM 再强，它还是有局限的。**
>
> 看 Cell 0 列的三个例子。你问「今天北京天气怎么样？」——LLM 说不知道，因为它的知识是冻结的。你说「帮我算 12345 × 67890」——LLM 可能算错，因为它本质上不是计算器。你说「帮我发邮件」——更做不到，因为它不能操作外部系统。
>
> 怎么办？两个解决方案。第一个叫 **Agent**——给 LLM 配工具，让它能调用搜索引擎、计算器、API。第二个叫 **RAG**——让 LLM 能查外部知识库，相当于给它「开卷考试」的能力。
>
> 先运行 Cell 3 准备环境。（运行 Cell 3）看到「环境准备完成！」就好。
>
> 现在运行 Cell 4 看 Agent 架构全景图。（运行 Cell 4）大家看这张图——中间红色圆圈是 **LLM Core**，负责推理和生成。上方虚线框是**工具层**（Calculator、Search、Database、API），左下是**记忆系统**，右下是**规划系统**。中间还有三个小标签——Thought、Action、Observation，这就是 ReAct 循环，等一下重点讲。
>
> 输出末尾总结了四个核心组件：LLM Core、Tools、Memory、Planning。今天我们主要聚焦前两个——LLM 如何调用工具，以及如何通过 RAG 检索知识。

👀 输出要点
- Cell 3：`环境准备完成！`
- Cell 4：Agent 架构全景图可视化 + 文字总结
  - 4 个核心组件：LLM Core / Tools / Memory / Planning
  - ReAct 循环：Thought → Action → Observation → ...

❓ 预判问题
- **Q：Agent 和 RAG 有什么区别？**
  A：Agent 是一个更大的概念——LLM + 工具 + 规划能力。RAG 是一种特定技术——通过检索外部文档来增强生成。RAG 可以作为 Agent 的一个「工具」来使用，后面 Cell 20 会演示。
- **Q：这跟 ChatGPT 的插件是一回事吗？**
  A：对！ChatGPT Plugins 就是 Agent 架构的商业化实现——LLM 决定调用哪个插件，传什么参数，然后基于结果回答。

➡️ 转场

> 全景图有了，下面我们深入第一个核心概念——ReAct 范式。这是 Agent 的灵魂。

---

## 第二段：ReAct 范式——理论 + 示例演示（Cell 5-7）

📍 浏览 Cell 5-6（Markdown 理论）、运行 Cell 7（ReAct 示例）

⏱ 时间分配：14 分钟（理论 9 分钟 + 示例 5 分钟）

🎯 本段目标
- 理解 ReAct 循环：Thought → Action → Observation → Thought → ... → Final Answer
- 理解为什么 Thought 步骤比直接调工具更好
- 了解 Agent 的局限性和架构演进

🗣 讲课话术

> 先看 Cell 5。Agent = LLM + 工具调用能力 + 规划能力。ReAct 是让这三者协作的框架，来自 Yao 等人 2022 年的论文。
>
> ReAct 把**推理（Reasoning）** 和 **行动（Acting）** 交织在一起。以前要么只推理不行动（Chain-of-Thought），要么只行动不推理（直接调工具）。ReAct 的做法是：先想清楚（Thought），再动手（Action），然后看结果（Observation），再想、再动。
>
> 看 Cell 6 的详细示例。用户问「湖北鄂州天气如何？」模型先 Thought：「我需要查找天气信息，因为我的训练数据不包含实时数据。」——注意，这个思考步骤很关键！它迫使模型**想清楚为什么要调工具**。然后 Action：`search("湖北鄂州 天气")`。Observation 返回「25°C，小雨，湿度 80%」。再 Thought：「有雨且湿度高，应建议带伞。」最后给出 Final Answer。
>
> **没有 Thought 步骤会怎样？** 模型可能直接跳到调用错误的工具，或者拿到结果后不知道怎么整合。Thought 步骤迫使模型「想清楚再做」。
>
> 再看 Cell 6 里的一个重要问题：**为什么 Agent 需要结构化输出？** LLM 生成的是自由文本，但工具调用需要精确的参数。比如 LLM 说「我觉得应该搜一下天气」——这没法执行。必须变成 `{"tool": "search", "args": {"query": "鄂州天气"}}`。这就是 Function Calling 的本质——用训练让模型在特定场景下只输出合法 JSON。
>
> Cell 6 底部有个 Agent 局限性的表格，大家标记一下。最常见的问题是**工具调用幻觉**——模型编造不存在的工具。还有**无限循环**——反复执行同一步骤。以及**上下文溢出**——多轮调用后对话太长。
>
> 最后看 Agent 架构演进——从 Level 1 单次调用，到 Level 2 ReAct 循环（今天主要实现这个），到 Level 3 多 Agent 协作（AutoGen、CrewAI），到 Level 4 自主 Agent（Devin、Manus）。
>
> 好，运行 Cell 7 看具体示例。（运行 Cell 7）这就是一个完整的 ReAct 流程。问题是天气查询，经过一轮 Thought-Action-Observation，最后给出 Final Answer：「湖北鄂州当前 25°C 有雨，湿度 80%，建议带伞。」

👀 输出要点
- Cell 7 完整 ReAct 示例：
  - Question: 湖北鄂州天气如何？
  - Thought → Action: get_weather(city="湖北鄂州")
  - Observation: {temperature: 25, condition: "Rainy", humidity: 80}
  - Thought → Final Answer: 25°C 有雨，建议带伞

❓ 预判问题
- **Q：ReAct 和 Chain-of-Thought（CoT）有什么区别？**
  A：CoT 只推理不行动——模型在内部想完所有步骤后直接输出答案。ReAct 每次推理后可以调工具获取外部信息。CoT 适合纯推理题，ReAct 适合需要外部信息的任务。
- **Q：Thought 步骤是必须的吗？**
  A：理论上不是，但实验证明有 Thought 的 Agent 显著优于直接 Action 的。就像人做事前想一想总比直接冲好。
- **Q：Level 3 多 Agent 协作是什么意思？**
  A：比如一个 Agent 负责搜索、一个负责写代码、一个负责审核，它们之间互相传递消息协作完成复杂任务。像 CrewAI、AutoGen 就是这类框架。

➡️ 转场

> ReAct 的循环理解了，那 Action 具体怎么调用工具？下一节我们来看 Tool Use——如何定义工具、如何让 LLM 调用函数。

---

## 第三段：Tool Use——工具定义与调用（Cell 8-10）

📍 浏览 Cell 8（Markdown 说明）、运行 Cell 9（工具定义 + 测试）、运行 Cell 10（工具描述）

⏱ 时间分配：13 分钟

🎯 本段目标
- 理解工具的三要素：名称、参数说明、返回值说明
- 看到工具就是普通的 Python 函数
- 理解工具注册表（TOOLS 字典）的作用

🗣 讲课话术

> Cell 8 告诉我们，工具就是 Python 函数，需要三样东西：清晰的名称、参数说明、返回值说明。这跟大家平时写函数文档一样，只不过这些文档是给 LLM 看的——LLM 要通过描述来决定什么时候调用哪个工具。
>
> 运行 Cell 9。（运行 Cell 9）我们定义了四个工具：
> - `calculator`：计算数学表达式，内部用 `eval()` 实现
> - `search`：模拟搜索引擎，其实就是一个关键词匹配的字典
> - `get_current_time`：获取当前时间
> - `get_weather`：模拟天气查询
>
> 看测试输出——`calculator('2 + 3 * 4') = 14`，计算正确。`search('湖北鄂州天气') = 湖北鄂州天气：25°C，小雨，湿度 80%`，命中了模拟数据。`get_current_time()` 返回当前时间戳。
>
> 注意底部的 `TOOLS` 字典——这就是**工具注册表**。Agent 执行 Action 时，会从这个字典里查找对应的函数来调用。名字对不上就报错。
>
> 运行 Cell 10 看工具描述文本。（运行 Cell 10）这段文本会被放进 Agent 的 system prompt 里，告诉 LLM「你有这些工具可以用」。大家注意，每个工具都有名称、参数类型、返回值说明和示例。这个描述质量直接决定 LLM 能不能正确调用工具。描述写得模糊，LLM 就可能调错。
>
> 大家想一下：如果你要加一个新工具，比如「翻译」，你需要做什么？（等 3 秒）对——写一个 Python 函数，加上 docstring，然后注册到 TOOLS 字典里。就这么简单。

👀 输出要点
- Cell 9 工具测试：
  - `calculator('2 + 3 * 4') = 14`
  - `search('湖北鄂州天气') = 湖北鄂州天气：25°C，小雨，湿度 80%。`
  - `get_current_time()` = 当前时间字符串
- Cell 10：三个工具的描述文本，含参数类型和示例

❓ 预判问题
- **Q：用 eval() 计算表达式安全吗？**
  A：教学环境可以，但生产环境绝对不行！用户可以注入恶意代码。实际应该用 `ast.literal_eval` 或专门的数学解析库（如 `sympy`）。
- **Q：真实的搜索工具怎么实现？**
  A：调用搜索引擎 API（如 Google Search API、Bing API），或者用 `requests` 库调用自己的后端搜索服务。这里用模拟字典是为了教学不依赖外部服务。
- **Q：工具描述为什么这么重要？**
  A：因为 LLM 完全依靠工具描述来决定什么时候用什么工具。描述不清楚，LLM 就不知道这个工具能做什么，可能该调的时候不调，不该调的时候瞎调。

➡️ 转场

> 工具定义好了，接下来的关键问题是：LLM 怎么告诉系统「我要调这个工具」？答案是 Function Calling 格式。

---

## 休息 + 回顾（第 35-40 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. LLM 有三大局限：知识冻结、计算不精确、无法操作外部系统。Agent（工具调用）和 RAG（知识检索）是两大解决方案。
2. ReAct 范式将推理和行动交织：Thought → Action → Observation → ... → Final Answer。Thought 步骤迫使模型「想清楚再做」。
3. 工具就是 Python 函数，通过工具注册表（TOOLS 字典）供 Agent 调用。工具描述的质量直接决定 LLM 能否正确使用工具。

**下一段预告：** 我们要实现一个完整的 Agent——先看 Function Calling 的标准 JSON 格式，然后构建一个 ProductionAgent 类，让它真正跑起来。

---

## 第四段：Agent 实现——Function Calling + ProductionAgent（Cell 11-13）

📍 浏览 Cell 11（Markdown），运行 Cell 12（Function Calling 格式详解），运行 Cell 13（ProductionAgent 完整实现与测试）

⏱ 时间分配：15 分钟（Function Calling 5 分钟 + Agent 实现 10 分钟）

🎯 本段目标
- 掌握 OpenAI 风格 Function Calling 的三阶段 JSON 格式
- 理解 ProductionAgent 的完整 ReAct 循环实现
- 看到 Agent 实际运行的过程

🗣 讲课话术

> 运行 Cell 12。（运行 Cell 12）这段代码展示了 OpenAI 风格 Function Calling 的标准格式，有三个阶段。
>
> **第一阶段：工具定义。** 用 JSON Schema 描述工具参数。大家看 `get_weather` 工具——`city` 参数是 string 类型，描述是「城市名称」，`unit` 有枚举约束 `["celsius", "fahrenheit"]`，`required` 里只有 `city`。这个 Schema 会被发给 LLM，让它知道怎么填参数。
>
> **第二阶段：LLM 返回工具调用。** 看输出里 `tool_calls` 数组——`id` 是调用标识，`function.name` 是 `get_weather`，`arguments` 是 JSON 字符串 `{"city": "湖北鄂州", "unit": "celsius"}`。注意 `content` 是 `null`——说明这次 LLM 不是在说话，而是在调用工具。
>
> **第三阶段：工具执行结果。** `role` 变成了 `tool`，通过 `tool_call_id` 关联回去，`content` 是返回的 JSON。
>
> 底部 4 个关键要点记一下：(1) 工具用 JSON Schema 描述参数；(2) LLM 返回结构化的 tool_calls；(3) 结果通过 tool_call_id 关联；(4) 这是 OpenAI/Claude 等 API 的标准格式。
>
> 好，现在看重头戏——运行 Cell 13。（运行 Cell 13）这是一个完整的 ProductionAgent 类，大约 180 行代码。代码很长但结构很清晰，我带大家走一遍核心逻辑。
>
> 1. `__init__`：注册工具、设最大步数（10）、构建 system prompt
> 2. `_build_system_prompt`：把所有工具的 docstring 组织成提示词
> 3. `parse_response`：用正则表达式解析 LLM 输出——检测 `Final Answer:` 或 `Action: tool_name(args)`
> 4. `execute_tool`：查 TOOLS 字典，调用函数
> 5. `simulate_llm`：这里是规则模拟，不是真正的 LLM。实际使用时替换成 API 调用
> 6. `run`：主循环——反复调用 LLM → 解析 → 如果是 Action 就执行工具 → 直到 Final Answer 或达到最大步数
>
> 看测试 1 的输出。问题是「请计算 15 * 7 + 23」。步骤 1：模拟 LLM 输出 Thought「这是数学问题」→ Action `calculator(expression="15 * 7 + 23")`，执行后观察结果 **128**。步骤 2：LLM 看到 Observation，输出 Final Answer「结果是 128」。两步完成！
>
> 测试 2 是一般问题「什么是 Python 编程语言？」。模拟 LLM 判断不需要工具，直接给 Final Answer，一步完成。
>
> 大家注意 `max_steps = 10` 这个安全阀——防止 Agent 进入无限循环。这就是 Cell 6 提到的局限性之一的解决方案。

👀 输出要点
- Cell 12 Function Calling 三阶段 JSON 格式：
  - 工具定义（JSON Schema）
  - LLM 工具调用（tool_calls, id: call_abc123）
  - 工具结果（role: tool, tool_call_id 关联）
- Cell 13 测试 1（数学计算）：
  - 步骤 1：Thought → Action: calculator → Observation: **128**
  - 步骤 2：Thought → Final Answer: 结果是 128
- Cell 13 测试 2（一般问题）：
  - 步骤 1：直接 Final Answer（不需要工具）

❓ 预判问题
- **Q：simulate_llm 是真正的 LLM 吗？**
  A：不是！这是规则模拟——用 if-else 判断关键词来决定输出。真正的 Agent 会调用 OpenAI/Claude 的 API，让真实 LLM 来决定。但核心流程（parse → execute → loop）是完全一样的。
- **Q：parse_response 为什么用正则表达式？**
  A：因为我们用的是 ReAct 文本格式（`Action: tool_name(args)`）。如果用 OpenAI 的 Function Calling API，模型直接返回 JSON，不需要正则解析。
- **Q：max_steps = 10 够用吗？**
  A：简单任务 2-3 步就够了。复杂任务（比如多步搜索 + 计算）可能需要 5-8 步。10 步是合理的安全上限。如果超过 10 步还没答案，通常说明 Agent 卡住了。

➡️ 转场

> Agent 部分讲完了。现在进入另一个重要话题——RAG，检索增强生成。如果说 Agent 是给 LLM 装了手和脚，RAG 就是给 LLM 装了一个外部大脑。

---

## 第五段：RAG 理论 + 向量检索实现 + 可视化（Cell 14-18）

📍 浏览 Cell 14-16（Markdown 理论），运行 Cell 17（VectorRAG 实现 + 检索测试 + 相似度可视化），运行 Cell 18（RAG 流程图）

⏱ 时间分配：15 分钟（理论 7 分钟 + 代码 8 分钟）

🎯 本段目标
- 理解 RAG 的核心思路：先检索再生成
- 掌握余弦相似度公式和直觉
- 看到完整的 RAG 流程：添加文档 → 向量化 → 检索 → 构建 Prompt → 生成
- 理解稠密检索 vs 稀疏检索的区别

🗣 讲课话术

> Cell 14 介绍了 RAG 的全貌。**RAG = Retrieval-Augmented Generation，检索增强生成。** 核心思路四步：用户提问 → 从知识库检索相关文档 → 拼进 Prompt → LLM 基于文档回答。
>
> 打个比方：纯 LLM 是闭卷考试，RAG 是开卷考试。你不需要把所有知识都背在脑子里，只要能快速翻到正确的那一页。
>
> 为什么需要 RAG？三个原因。Cell 15 的表格讲得很清楚——**知识截止**（模型不知道最新信息）、**事实幻觉**（模型可能编造事实）、**私有数据不可达**（模型无法访问你们公司内部文档）。RAG 用检索来补这三个短板。
>
> Cell 15 还有一个数学公式值得看：$P(y|x) = \sum_{d} P(y|x,d) \cdot P(d|x)$。简单说就是——回答的概率等于「检索到某文档的概率」乘以「基于该文档生成回答的概率」，对所有文档求和。实际我们用 Top-K 近似，只取最相关的 K 篇文档。
>
> Cell 15 还对比了**稠密检索 vs 稀疏检索**。稀疏检索（BM25）基于词频，「汽车」和「轿车」被视为不同词。稠密检索（Embedding）把文本映射到向量空间，「汽车」和「轿车」的向量很接近。实际系统常两者结合。
>
> 好，重点来了——Cell 16 的**余弦相似度**公式。$\cos(a, b) = \frac{a \cdot b}{|a| \times |b|}$。值域 [-1, 1]。1 表示方向完全相同，0 表示正交，-1 表示完全相反。**只看方向不看长度**——这很重要，因为文档长短不一，我们只关心语义方向是否一致。
>
> 现在运行 Cell 17。（运行 Cell 17）先看上半部分——我们往知识库添加了 **12 个文档**，涵盖 Python、机器学习、PyTorch、Transformer、GPT、RAG、向量数据库、LangChain 等。
>
> 然后查询「PyTorch 是什么，由谁开发？」。看检索结果——Top 3 文档的相似度分别是多少？第一名是「PyTorch 是 Facebook AI Research 开源的深度学习框架，支持动态计算图」，相似度最高。第二名是「PyTorch 提供 GPU 加速张量与自动求导」。检索很准！
>
> 再看下方的**相似度条形图**。查询是「深度学习框架 PyTorch」，Top 5 文档按相似度排列。可以看到前两名明显高于其他文档——这就是向量检索的效果，语义相近的文档会被排在前面。
>
> 运行 Cell 18 看 RAG 流程图。（运行 Cell 18）从左到右：用户问题 → 检索器（向量搜索） → Top-K 文档 → LLM（生成器） → 回答。底部还有知识库反向连到检索器。这就是 RAG 的完整数据流。
>
> 注意：我们这里的 Embedding 是**模拟的**——用随机种子对每个词生成伪向量，然后取平均。真实项目用 sentence-transformers 或 OpenAI Embedding API。但检索流程和相似度计算逻辑完全一样。

👀 输出要点
- Cell 17：
  - 添加了 12 个文档到知识库
  - 查询「PyTorch 是什么，由谁开发？」Top-3 检索结果 + 相似度分数
  - Prompt 长度和参考文档数
  - 模拟 LLM 回答
  - 相似度条形图可视化（查询「深度学习框架 PyTorch」的 Top-5）
- Cell 18：RAG 流程图（用户问题 → 检索 → Top-K → LLM → 回答）

❓ 预判问题
- **Q：为什么用余弦相似度而不是欧氏距离？**
  A：余弦相似度只看方向不看长度，更适合文本场景——长文档和短文档的向量大小不同，但语义方向可能一致。欧氏距离受向量长度影响较大。
- **Q：Top-K 的 K 怎么选？**
  A：通常 3-5。K 太小可能漏掉关键信息，K 太大会引入噪声且占用 LLM 上下文窗口。还可以加一个相似度阈值，低于阈值的不要。
- **Q：chunk_size 怎么选？**
  A：Cell 15 讲了——经验值 512-1024 token，overlap 10-20%。太小上下文不够，太大噪声太多。LangChain 默认用递归切分。
- **Q：文档切分对检索有多大影响？**
  A：影响很大！一个讲 PyTorch 安装的段落和一个讲 PyTorch 性能优化的段落，如果混在一起，检索质量会下降。好的切分策略是 RAG 工程中最重要的优化点之一。

➡️ 转场

> RAG 的检索和生成都理解了。那 Agent 和 RAG 能不能结合起来？当然可以——RAG 可以作为 Agent 的一个工具！

---

## 休息 + 回顾（第 70-75 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. Function Calling 是 Agent 调用工具的标准接口——用 JSON Schema 定义工具，LLM 返回结构化的 tool_calls，结果通过 tool_call_id 关联。
2. ProductionAgent 的核心是一个 while 循环：调用 LLM → 解析响应 → 如果是 Action 就执行工具 → 将 Observation 加回对话 → 继续循环直到 Final Answer。
3. RAG 先检索后生成。余弦相似度 $\cos(a,b) = \frac{a \cdot b}{|a| |b|}$ 是向量检索的核心度量，只看方向不看长度。

**下一段预告：** 我们要把 Agent 和 RAG 结合起来——让 Agent 把 RAG 作为一个工具来使用，然后回顾本章总结和常见面试题。

---

## 第六段：Agent + RAG 结合 + 总结（Cell 19-22）

📍 浏览 Cell 19（Markdown），运行 Cell 20（RAGAgent 实现与测试），浏览 Cell 21-22（总结与下一步）

⏱ 时间分配：7 分钟

🎯 本段目标
- 看到 Agent + RAG 结合的实现方式
- 理解「RAG 作为 Agent 工具」的设计模式
- 回顾本章所有核心概念

🗣 讲课话术

> 运行 Cell 20。（运行 Cell 20）这段代码很精炼。RAGAgent 有两个工具：`search_knowledge_base` 和 `calculator`。其中 `search_knowledge_base` 内部调用 SimpleRAG 做检索——RAG 被包装成了 Agent 的一个工具！
>
> 看运行结果。问题是「请介绍一下 PyTorch」。Agent 的 Thought：「我应该查询知识库获取信息。」Action：`search_knowledge_base(query="请介绍一下 PyTorch")`。Observation 返回两个检索结果——「PyTorch 是一个开源的机器学习框架，支持动态计算图」和「PyTorch 常用于深度学习研究与工业落地」。然后 Agent 整合信息给出 Final Answer。
>
> 这就是 Agent + RAG 的核心思路：**Agent 决定什么时候需要查知识库，RAG 负责检索，LLM 负责整合回答。** 遇到计算题就调 calculator，遇到知识题就调 RAG。
>
> 大家看 Cell 21 的总结图谱。左边是 Agent 的 ReAct 循环（Thought → Action → Observe），右边是 RAG 的流程（问题 → Embedding → 向量检索 → 文档 + 问题 → LLM → 回答）。底部是 Tool Use 层。
>
> Cell 21 底部有四道面试题，值得看一下：
> - Q1：ReAct vs CoT——CoT 只推理，ReAct 推理 + 行动
> - Q2：稠密 vs 稀疏检索——BM25 快但不懂语义，Embedding 懂语义但需训练
> - Q3：Chunking 策略——512-1024 token，overlap 10-20%
> - Q4：Agent 失败模式——幻觉、无限循环、错误恢复、上下文溢出
>
> 最后，Cell 22 告诉我们——恭喜完成全部主线章节！从 Ch0 到 Ch12，我们走完了从零构建 LLM 的全流程。

👀 输出要点
- Cell 20 RAGAgent 测试：
  - Thought → Action: search_knowledge_base
  - Observation: 检索到两篇 PyTorch 相关文档
  - Final Answer: PyTorch 是一个机器学习框架，支持 GPU 加速与自动求导
- Cell 21：核心概念图谱、关键公式速查表、四道面试题

❓ 预判问题
- **Q：SimpleRAG 和前面的 VectorRAG 有什么区别？**
  A：SimpleRAG 用的是词重叠（稀疏检索），VectorRAG 用的是向量余弦相似度（稠密检索）。SimpleRAG 更简单但不理解语义，适合快速演示。
- **Q：实际产品中 Agent + RAG 是怎么部署的？**
  A：典型架构是 LangChain/LlamaIndex 做编排，Pinecone/Chroma/Milvus 做向量存储，OpenAI/Claude API 做 LLM。前端一个聊天界面，后端 Agent 根据用户输入决定是直接回答还是先检索。

➡️ 转场

> 核心内容讲完了。最后我们做一个动手练习——实现余弦相似度检索函数。

---

## 第七段：练习——实现余弦相似度检索（Cell 23-26）

📍 浏览 Cell 23-24（Markdown 练习说明），Cell 25（练习代码——学生填写），运行 Cell 26（验证测试）

⏱ 时间分配：8 分钟

🎯 本段目标
- 学生动手实现余弦相似度函数
- 理解向量检索的代码实现
- 通过测试验证正确性

🗣 讲课话术

> Cell 24 给出了练习要求：实现 `cosine_similarity(vec_a, vec_b)` 函数。公式我们已经讲过——点积除以两个范数的乘积。
>
> Cell 25 有代码骨架，三个 TODO 需要填：(1) 用 `np.dot` 算点积；(2) 用 `np.linalg.norm` 算范数；(3) 组合成余弦相似度，注意加 epsilon 防止除零。
>
> 大家先自己试 2 分钟。

**提示节奏**
- 0-2 分钟：自己思考和编码
- 2 分钟第一提示：三步走——`np.dot(vec_a, vec_b)` 算点积，`np.linalg.norm(vec_a)` 算范数，然后相除
- 4 分钟关键代码：
  ```python
  dot_product = np.dot(vec_a, vec_b)
  norm_a = np.linalg.norm(vec_a)
  norm_b = np.linalg.norm(vec_b)
  similarity = dot_product / (norm_a * norm_b + epsilon)
  ```

**常见错误**
- 忘记加 `epsilon`（零向量时会除零报错 NaN/Inf）
- 用 `np.linalg.norm` 时传了错误的参数（应该分别对 vec_a 和 vec_b 各算一次）
- 把点积算成了逐元素乘法（`np.multiply` 返回数组，不是标量）

**验证标准**
- Cell 26 会运行 6 个测试：
  1. 相同方向 → 相似度 ≈ 1.0
  2. 相反方向 → 相似度 ≈ -1.0
  3. 缩放不变性 → cos(a, 2a) = 1.0
  4. 近正交 → 相似度 ≈ 0
  5. 值域检查 → 结果在 [-1, 1]
  6. 零向量鲁棒性 → 不产生 NaN
- 最后还有检索 sanity check：加噪声的 query 向量 → Top-1 应命中原始文档
- 看到 `余弦相似度函数通过所有 test cases` 和 `检索 sanity check 通过` 即成功

❓ 预判问题
- **Q：为什么 epsilon 用 1e-8 而不是 0？**
  A：防止零向量（全零）时分母为 0 导致 NaN。1e-8 足够小不影响正常计算，但能避免数值异常。
- **Q：search_documents 函数的 top_k 是怎么实现的？**
  A：对所有文档计算相似度 → 按分数降序排序 → 取前 K 个。实际大规模系统用 FAISS 的近似最近邻（ANN）索引加速，不需要全部计算。
- **Q：检索 sanity check 中为什么要加 0.1 倍的噪声？**
  A：模拟真实场景——用户的查询不会和文档完全一样，但语义应该接近。0.1 倍噪声让 query 和原文档向量的余弦相似度约 0.9965，很高但不是 1.0。

➡️ 转场

> 很好！大家都通过了测试。这个余弦相似度函数就是 RAG 系统的核心引擎——无论你用 FAISS、Milvus 还是 Chroma，底层都在做这件事。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，运行环境准备 + Agent 架构全景图 | 0-4 |
| 8 | ReAct 范式理论 + 示例演示 | 5-7 |
| 22 | Tool Use：工具定义与调用 | 8-10 |
| 35 | **休息** | -- |
| 40 | Function Calling 格式 + ProductionAgent 实现 | 11-13 |
| 55 | RAG 理论 + VectorRAG 实现 + 可视化 | 14-18 |
| 70 | **休息** | -- |
| 75 | Agent + RAG 结合 + 本章总结 | 19-22 |
| 82 | 练习：余弦相似度实现 | 23-26 |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\text{cosine\_sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{|\vec{a}| \times |\vec{b}|}$$

$$P(y|x) = \sum_{d \in \mathcal{D}} P(y|x, d) \cdot P(d|x) \approx \sum_{d \in \text{Top-K}(x)} P(y|x, d) \cdot P(d|x)$$

### ReAct 循环

```
Thought → Action → Observation → Thought → ... → Final Answer
```

### Agent 核心组件

| 组件 | 作用 |
|:---|:---|
| LLM Core | 推理、生成 |
| Tools | 扩展外部能力（搜索、计算、API） |
| Memory | 上下文管理、知识存储 |
| Planning | 任务分解、执行策略 |

### 工具测试输出

| 工具调用 | 返回值 |
|:---|:---|
| `calculator('2 + 3 * 4')` | `14` |
| `calculator('15 * 7 + 23')` | `128` |
| `search('湖北鄂州天气')` | `湖北鄂州天气：25°C，小雨，湿度 80%。` |

### RAG 知识库

| 指标 | 值 |
|:---|:---|
| 文档数 | 12 |
| Embedding 维度（模拟） | 64 |
| 默认 Top-K | 3 |

### 检索对比

| 方法 | 代表 | 优点 | 缺点 |
|:---|:---|:---|:---|
| 稀疏检索 | BM25 | 快、无需训练 | 不理解语义 |
| 稠密检索 | Embedding | 理解语义 | 需训练、需 ANN 索引 |
| 混合检索 | BM25 + Embedding | 兼顾速度和精度 | 需要调权重 |

### 文档切分经验值

| 参数 | 推荐范围 |
|:---|:---|
| chunk_size | 512-1024 token |
| chunk_overlap | 10%-20% |

### Agent 架构演进

| 级别 | 说明 | 代表 |
|:---|:---|:---|
| Level 1 | 单次工具调用 | ChatGPT Plugins |
| Level 2 | ReAct 循环 | LangChain Agent |
| Level 3 | 多 Agent 协作 | AutoGen、CrewAI |
| Level 4 | 自主 Agent | Devin、Manus |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** `ModuleNotFoundError: No module named 'numpy'`

**应对：**
1. 在终端运行 `pip install numpy matplotlib`
2. 重启 Kernel 后重新运行
3. 本章不需要 PyTorch、不需要 GPU

### 场景 2：中文字体不显示

**症状：** Agent 架构全景图（Cell 4）或 RAG 流程图（Cell 18）中中文显示为方框

**应对：**
1. 确认 `assets/fonts/NotoSansCJKsc-Regular.otf` 文件存在
2. 如果不存在，从 Google Fonts 下载 Noto Sans CJK SC
3. 或修改 `plt.rcParams["font.sans-serif"]` 为系统中已有的中文字体
4. 可视化不影响核心教学——如果字体问题无法解决，跳过图形直接讲代码输出

### 场景 3：VectorRAG 检索结果不理想

**症状：** 检索到的文档和查询不太相关

**应对：**
1. 这是正常的——我们用的是**模拟 Embedding**（伪随机词向量取平均），不是真正的语义模型
2. 向学生解释：真实系统用 sentence-transformers 或 OpenAI Embedding 会好很多
3. 重点讲解流程和代码逻辑，不纠结于模拟检索的精确度

### 场景 4：练习代码（Cell 25-26）测试失败

**症状：** 余弦相似度测试不通过

**应对：**
1. 检查是否忘记加 `epsilon`（零向量测试会失败）
2. 检查 `np.dot` 是否正确使用（应返回标量，不是数组）
3. 检查范数计算是否分别对 `vec_a` 和 `vec_b` 各算一次
4. 确认公式：`dot_product / (norm_a * norm_b + epsilon)`

### 场景 5：学生对 Agent 模拟方式有疑问

**症状：** 「这不是真正的 AI 在思考吧？」

**应对：**
1. 直说：「对，simulate_llm 是 if-else 规则，不是真正的 LLM。」
2. 但强调：「核心框架（parse → execute → loop）和真实 Agent 完全一致。把 simulate_llm 换成 OpenAI API 调用就是真正的 Agent。」
3. 如果有 API Key，可以现场演示替换为真实 API（但这不是必需的）

### 场景 6：时间不够

**可以跳过的内容（按优先级）：**
1. Cell 15 RAG 深层理论 Markdown 中的 BM25 公式和 Reranking 细节——口头带过即可
2. Cell 18 RAG 流程图——用 Cell 14 的文字描述替代
3. Cell 12 Function Calling 格式详解——简化为「LLM 输出 JSON 格式的工具调用」

**不能跳过的内容：**
1. ReAct 循环概念（Cell 5-7）——这是 Agent 的灵魂
2. 工具定义和测试（Cell 9）——动手环节
3. ProductionAgent 运行演示（Cell 13）——看到 Agent 真正跑起来
4. 余弦相似度概念（Cell 16）——RAG 的核心
5. Agent + RAG 结合（Cell 20）——串联全章